# Notebook 1 — Problem Setup & Baselines

## Story
We tackle **multi-label image classification** across 12 object categories.
An image may contain any combination of objects, so each prediction is a
binary vector of length 12 rather than a single class index.

In this notebook we:
1. Explore the dataset (class distribution, multi-label statistics, sample images).
2. Define the evaluation metrics we will use throughout the project.
3. Establish **non-learned baselines** (random and frequency-based predictors)
   to understand the lower bound any model must beat.

**Labels (12):** `pen, paper, book, clock, phone, laptop, chair, desk, bottle, keychain, backpack, calculator`

## 1. Imports & Setup

In [ ]:
import sys
sys.path.insert(0, "../")
sys.path.insert(0, "../experiments")

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

from eval import LABEL_ORDER
from utils import (
    set_seed, load_dataset, split_dataset,
    get_train_transform, get_eval_transform, build_dataloaders,
    plot_per_class_examples, plot_multilabel_examples,
    run_baselines, NUM_LABELS,
)

SEED   = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Labels ({NUM_LABELS}): {LABEL_ORDER}")

## 2. Config

In [ ]:
BASE_DATA_DIR = "../data/aggregated"
IMAGE_SIZE    = 128
BATCH_SIZE    = 256
SPLIT         = [0.8, 0.1, 0.1]

## 3. Load Dataset & Split

In [ ]:
full_dataset = load_dataset(BASE_DATA_DIR)
train_raw, val_raw, test_raw = split_dataset(full_dataset, SPLIT, SEED)

train_transform = get_train_transform(IMAGE_SIZE)
eval_transform  = get_eval_transform(IMAGE_SIZE)
train_loader, val_loader, test_loader = build_dataloaders(
    train_raw, val_raw, test_raw, train_transform, eval_transform, BATCH_SIZE
)

print(f"Total samples  : {len(full_dataset)}")
print(f"Train / Val / Test : {len(train_raw)} / {len(val_raw)} / {len(test_raw)}")

## 4. EDA — Dataset Statistics

Before building any model, let's understand the data:
- How many images contain each label?
- How many labels does the average image have?
- Are the classes balanced?

In [ ]:
# Gather all targets from the full dataset
all_targets = torch.stack([full_dataset[i][1] for i in range(len(full_dataset))])

# --- Per-class prevalence ---
prevalence = all_targets.mean(dim=0).numpy()
fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(LABEL_ORDER, prevalence, color="steelblue")
ax.set_ylabel("Fraction of images containing label")
ax.set_title("Per-class prevalence in the full dataset")
ax.set_xticklabels(LABEL_ORDER, rotation=35, ha="right")
for bar, val in zip(bars, prevalence):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{val:.2f}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

print("\nPer-class label count:")
counts = all_targets.sum(dim=0).int().numpy()
for lbl, cnt in sorted(zip(LABEL_ORDER, counts), key=lambda x: -x[1]):
    print(f"  {lbl:<14}  {cnt:>5}  ({100*cnt/len(full_dataset):.1f}%)")

In [ ]:
# --- Label-count distribution (how many labels per image?) ---
labels_per_image = all_targets.sum(dim=1).int().numpy()
count_dist = Counter(labels_per_image.tolist())

fig, ax = plt.subplots(figsize=(8, 4))
xs = sorted(count_dist.keys())
ys = [count_dist[x] for x in xs]
ax.bar(xs, ys, color="darkorange")
ax.set_xlabel("Number of labels per image")
ax.set_ylabel("Number of images")
ax.set_title("Distribution of label count per image")
ax.set_xticks(xs)
plt.tight_layout()
plt.show()

print(f"\nAverage labels per image : {labels_per_image.mean():.2f}")
print(f"Max labels per image     : {labels_per_image.max()}")
print(f"Images with 1 label only : {(labels_per_image == 1).sum()}")
print(f"Images with >3 labels    : {(labels_per_image > 3).sum()}")

## 5. Sample Images — Per Class and Multi-Label

In [ ]:
plot_per_class_examples(train_raw, per_class=3)

In [ ]:
plot_multilabel_examples(train_raw, max_items=9)

## 6. Metrics Overview

Because multiple labels can be active simultaneously, standard accuracy is not
informative. We track:

| Metric | Description |
|---|---|
| **Exact Match** | Fraction of images where the predicted set equals the ground-truth set exactly. |
| **Hamming Accuracy** | Mean per-label accuracy (each of 12 labels treated independently). |
| **Mean IoU** | Jaccard index — intersection over union of predicted and true label sets. |
| **Micro Precision** | TP / (TP + FP) aggregated across all labels. |
| **Micro Recall** | TP / (TP + FN) aggregated across all labels. |
| **Micro F1** | Harmonic mean of micro precision and recall — **our primary metric**. |

Micro F1 is preferred because it accounts for class imbalance and rewards
correctly predicting both common and rare labels.

## 7. Non-Learned Baselines

We evaluate three frequency-based predictors and a random predictor.
These set the **floor** that any learned model must surpass.

- **Top-k frequency**: always predict the k most common labels.
- **Random**: sample each label independently with p=0.5.

In [ ]:
best_name, best_val_metrics, best_test_metrics = run_baselines(
    train_loader, val_loader, test_loader, NUM_LABELS, DEVICE
)

## 8. Takeaways

- The best non-learned baseline achieves F1 < ~0.5 despite the apparent simplicity.
- Multi-label classification is harder than single-label: errors on any one label
  reduce exact match to 0 for that sample.
- We need a model that **learns visual features** to distinguish object categories.

**Next:** We try building our own CNN from scratch (Notebook 2).